# Librerias Necesarias

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# Descarga del DataSet
- https://www.kaggle.com/datasets/drscarlat/melanoma

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download drscarlat/melanoma

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/melanoma.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import os
os.listdir('/content/DermMel')

# Creacion de los Generadores de imagenes

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Directorio base
data_dir = '/content/DermMel'

# Generadores de datos
datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

# Carga datos de entrenamiento (TRAIN)
train_generator = datagen.flow_from_directory(
    os.path.join(data_dir, 'train_sep'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

# Carga de datos de validacion (VALID)
validation_generator = datagen.flow_from_directory(
    os.path.join(data_dir, 'valid'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
)

# Carga de datos de prueba (TEST)
test_generator = datagen.flow_from_directory(
    os.path.join(data_dir, 'test'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Visualizacion de algunos datos

In [ ]:
# Verificar los labels
print(train_generator.class_indices)

In [ ]:
import matplotlib.pyplot as plt

# Obtener un lote de imagenes y etiquetas del generador de entrenamiento
images, labels = next(train_generator)

print(images.shape)
print(labels.shape)

# Obtener las clases y sus indices
class_indices = train_generator.class_indices
class_names = list(class_indices.keys())

num_images = 5

# Crear una figura para mostrar las imagenes
plt.figure(figsize=(15, 5))
for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[int(labels[i])])
    plt.axis('off')

plt.show()

# Preparacion del modelo

In [ ]:
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.initializers import he_normal, glorot_uniform

# Arquitectura con menos capas y LeakyReLu
modeloCNN = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128, 3), kernel_initializer=he_normal()),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', kernel_initializer=he_normal()),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer=glorot_uniform()),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Arquitectura con mas capas y funciones de activacion variadas
modeloCNN2 = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation=LeakyReLU(alpha=0.1), input_shape=(128,128, 3), kernel_initializer=he_normal()),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', kernel_initializer=he_normal()),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu', kernel_initializer=he_normal()),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(256, (3,3), activation='relu', kernel_initializer=glorot_uniform()), # capa adicional
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(250, activation='relu', kernel_initializer=he_normal()),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


In [ ]:
# Compilacion de los modelos
modeloCNN.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

modeloCNN2.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

## CNN 1

In [ ]:
tensorboardCNN = TensorBoard(log_dir='logs/cnn')
historyCNN = modeloCNN.fit(train_generator,
              epochs=10,
              validation_data=validation_generator,
              callbacks=[tensorboardCNN])

## CNN 2

In [ ]:
tensorboardCNN2 = TensorBoard(log_dir='logs/cnn2')
historyCNN2 = modeloCNN2.fit(train_generator,
              epochs=10,
              validation_data=validation_generator,
              callbacks=[tensorboardCNN2])

# Graficos

In [ ]:
# Función para graficar las pérdidas de entrenamiento y validación
def plot_loss(history, model_name):
    plt.plot(history.history['loss'], label='Entrenamiento')
    plt.plot(history.history['val_loss'], label='Validación')
    plt.xlabel('Épocas')
    plt.ylabel('Pérdida')
    plt.title(f'Pérdida vs Épocas ({model_name})')
    plt.legend()
    plt.show()

In [ ]:
# Gráfica de pérdidas para el modelo 1
plot_loss(historyCNN, 'Modelo CNN 1')

In [ ]:
# Gráfica de pérdidas para el modelo 2
plot_loss(historyCNN2, 'Modelo CNN 2')

# Evaluacion Final (Modelo 2: "modeloCNN2")

In [ ]:
test_loss, test_acc = modeloCNN2.evaluate(test_generator)
print(f"Pérdida en el conjunto de prueba: {test_loss}")
print(f"Exactitud en el conjunto de prueba: {test_acc * 100:.2f}%")

# ENTRENAMIENTO CON ESCALA B/N (blanco y negro)

## Modificacion de los datos

In [ ]:
# Generadores de datos en blanco y negro
datagen_bw = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

# Carga de datos de entrenamiento (TRAIN) en blanco y negro
train_generator_bw = datagen_bw.flow_from_directory(
    os.path.join(data_dir, 'train_sep'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    subset='training'
)

# Carga de datos de validacion (VALID) en blanco y negro
validation_generator_bw = datagen_bw.flow_from_directory(
    os.path.join(data_dir, 'valid'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
)

# Carga de datos de prueba (TEST) en blanco y negro
test_generator_bw = datagen_bw.flow_from_directory(
    os.path.join(data_dir, 'test'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False
)

In [ ]:
import matplotlib.pyplot as plt

# Obtener un lote de imagenes y etiquetas del generador de entrenamiento
images, labels = next(train_generator_bw)

print(images.shape)
print(labels.shape)

# Obtener las clases y sus indices
class_indices = train_generator_bw.class_indices
class_names = list(class_indices.keys())

num_images = 5

# Crear una figura para mostrar las imagenes
plt.figure(figsize=(15, 5))
for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[int(labels[i])])
    plt.axis('off')

plt.show()

## Ajuste en la entrada del modelo

In [ ]:
modeloCNN2_bw = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(256, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(250, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
modeloCNN2_bw.compile(optimizer='adam',
                      loss='binary_crossentropy',
                      metrics=['accuracy'])

In [ ]:
tensorboardCNN2_bw = TensorBoard(log_dir='logs/cnn2_bw')
history_bw = modeloCNN2_bw.fit(train_generator_bw,
                               epochs=10,
                               validation_data=validation_generator_bw,
                               callbacks=[tensorboardCNN2_bw])

## Pruebas

In [ ]:
test_loss_bw, test_acc_bw = modeloCNN2_bw.evaluate(test_generator_bw)
print(f"Pérdida en el conjunto de prueba: {test_loss_bw}")
print(f"Exactitud en el conjunto de prueba: {test_acc_bw * 100:.2f}%")

# Entrenamiento con Aumento de datos y B/W

## Modificacion de datos

In [ ]:
# Generadores de datos en blanco y negro
datagen_bw_augmented = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=[0.7, 1.4],
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,              # Rotación aleatoria hasta 20 grados
    width_shift_range=0.2,          # Desplazamiento horizontal aleatorio
    height_shift_range=0.2,         # Desplazamiento vertical aleatorio
    brightness_range=[0.8, 1.2]     # Rango de brillo
)

# Carga de datos de entrenamiento (TRAIN) en blanco y negro (aumentado)
train_generator_bw_aug = datagen_bw_augmented.flow_from_directory(
    os.path.join(data_dir, 'train_sep'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    subset='training'
)

# Carga de datos de validacion (VALID) en blanco y negro
validation_generator_bw_aug = datagen_bw_augmented.flow_from_directory(
    os.path.join(data_dir, 'valid'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
)

# Carga de datos de prueba (TEST) en blanco y negro
test_generator_bw_aug = datagen_bw_augmented.flow_from_directory(
    os.path.join(data_dir, 'test'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False
)

In [ ]:
import matplotlib.pyplot as plt

# Obtener un lote de imagenes y etiquetas del generador de entrenamiento
images, labels = next(train_generator_bw_aug)

print(images.shape)
print(labels.shape)

# Obtener las clases y sus indices
class_indices = train_generator_bw_aug.class_indices
class_names = list(class_indices.keys())

num_images = 5

# Crear una figura para mostrar las imagenes
plt.figure(figsize=(15, 5))
for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[int(labels[i])])
    plt.axis('off')

plt.show()

## Entrenamiento

In [ ]:
tensorboardCNN2_bw_aug = TensorBoard(log_dir='logs/cnn2_bw_aug')
history_bw_aug = modeloCNN2_bw.fit(train_generator_bw_aug,
                               epochs=10,
                               validation_data=validation_generator_bw_aug,
                               callbacks=[tensorboardCNN2_bw_aug])

## Pruebas:

In [ ]:
test_loss_bw_aug, test_acc_bw_aug = modeloCNN2_bw.evaluate(test_generator_bw_aug)
print(f"Pérdida en el conjunto de prueba: {test_loss_bw_aug}")
print(f"Exactitud en el conjunto de prueba: {test_acc_bw_aug * 100:.2f}%")

# Manejar la RAM

In [ ]:
import gc
gc.collect()

# TensorBoard

In [ ]:
%load_ext tensorboard

In [ ]:
%reload_ext tensorboard

In [ ]:
%tensorboard --logdir logs

# TEST

In [ ]:
from tensorflow.python.client import device_lib
print("Num of GPU's Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
!ls -la

In [ ]:
!rm -r logs/